In [ ]:
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# Load data
train_df = pd.read_csv('drive/MyDrive/Data/train.csv')
test_df = pd.read_csv('drive/MyDrive/Data/test.csv')

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
# Fill missing values
train_df['question1'] = train_df['question1'].fillna('')
train_df['question2'] = train_df['question2'].fillna('')
test_df['question1'] = test_df['question1'].fillna('')
test_df['question2'] = test_df['question2'].fillna('')

In [ ]:
# Preprocessing functions
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\W', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\b\w{1,2}\b', '', text)  # Remove short words
    text = ' '.join([lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words])
    return text


In [ ]:
# Clean text
train_df['question1'] = train_df['question1'].apply(clean_text)
train_df['question2'] = train_df['question2'].apply(clean_text)
test_df['question1'] = test_df['question1'].apply(clean_text)
test_df['question2'] = test_df['question2'].apply(clean_text)

In [ ]:
# TF-IDF vectorization
tfidf = TfidfVectorizer(max_features=5000)
tfidf_q1 = tfidf.fit_transform(train_df['question1'])
tfidf_q2 = tfidf.transform(train_df['question2'])

tfidf_q1_test = tfidf.transform(test_df['question1'])
tfidf_q2_test = tfidf.transform(test_df['question2'])

In [ ]:
# Function to compute cosine similarity in batches
def batch_cosine_similarity(matrix1, matrix2, batch_size=1000):
    num_batches = int(np.ceil(matrix1.shape[0] / batch_size))
    similarities = np.zeros(matrix1.shape[0])
    for i in range(num_batches):
        start = i * batch_size
        end = min((i + 1) * batch_size, matrix1.shape[0])
        similarities[start:end] = cosine_similarity(matrix1[start:end], matrix2[start:end]).diagonal()
    return similarities

In [ ]:
# Optimized cosine similarity
train_cosine_similarities = batch_cosine_similarity(tfidf_q1, tfidf_q2)
test_cosine_similarities = batch_cosine_similarity(tfidf_q1_test, tfidf_q2_test)

In [ ]:
train_df['cosine_similarity'] = train_cosine_similarities
test_df['cosine_similarity'] = test_cosine_similarities

In [ ]:
# Features and labels
X_train = train_df[['cosine_similarity']]
y_train = train_df['is_duplicate']
X_test = test_df[['cosine_similarity']]

In [ ]:
# Split data
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(X_train, y_train, test_size=0.2, random_state=42)


In [ ]:
# Initialize models
models = {
    'Logistic Regression': LogisticRegression(),
    'Random Forest': RandomForestClassifier(),
    'XGBoost': XGBClassifier()
}

In [ ]:
# Train and evaluate models
for name, model in models.items():
    model.fit(X_train_split, y_train_split)
    y_pred = model.predict(X_val_split)
    print(f'{name} Accuracy: {accuracy_score(y_val_split, y_pred)}')
    print(classification_report(y_val_split, y_pred))